# 2025 한국 반려동물 보고서 PDF → ChromaDB

PDF를 페이지별로 추출하고 청킹한 뒤 `pet_analysis` 컬렉션에 적재합니다.
노트북을 다시 실행해도 같은 PDF의 기존 청크를 먼저 제거하므로 중복 적재되지 않습니다.

In [10]:
from pathlib import Path
import hashlib
import re
import shutil

import pymupdf
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

PROJECT_DIR = Path.cwd().resolve().parent
PDF_PATH = PROJECT_DIR / "data" / "2025 한국 반려동물 보고서.pdf"
CHROMA_DIR = PROJECT_DIR / "data" / "chroma_db"
COLLECTION_NAME = "pet_analysis_1024"
EMBEDDING_MODEL = "BAAI/bge-m3"

assert PDF_PATH.exists(), f"PDF를 찾을 수 없습니다: {PDF_PATH}"
CHROMA_DIR.mkdir(parents=True, exist_ok=True)
print(f"PDF: {PDF_PATH}")
print(f"Chroma 경로: {CHROMA_DIR}")
print(f"컬렉션: {COLLECTION_NAME}")

PDF: C:\Users\Playdata\Desktop\mle-01-p1-team2\data\2025 한국 반려동물 보고서.pdf
Chroma 경로: C:\Users\Playdata\Desktop\mle-01-p1-team2\data\chroma_db
컬렉션: pet_analysis_1024


## 1. PDF 텍스트 추출

`page_chunks=True`를 사용해 페이지 경계를 보존합니다. 페이지 번호는 1부터 시작하도록 저장합니다.

In [11]:
pdf_document = pymupdf.open(str(PDF_PATH))
control_char_pattern = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")


def normalize_page_text(text: str) -> str:
    text = control_char_pattern.sub(" ", text)
    lines = [re.sub(r"[ \t]{2,}", " ", line).strip() for line in text.splitlines()]
    return "\n".join(line for line in lines if line)


pages = [
    {"text": normalize_page_text(page.get_text("text", sort=True)), "page": page_number}
    for page_number, page in enumerate(pdf_document, start=1)
]
pages = [page for page in pages if page["text"].strip()]

replacement_character = chr(0xFFFD)
replacement_count = sum(page["text"].count(replacement_character) for page in pages)
assert replacement_count == 0, f"깨진 문자(대체문자)가 {replacement_count:,}개 발견되었습니다."
long_space_count = sum(
    len(re.findall(r"[ \t]{10,}", page["text"])) for page in pages
)
assert long_space_count == 0, f"10칸 이상 연속 공백이 {long_space_count:,}개 발견되었습니다."

print(f"텍스트가 있는 페이지 수: {len(pages):,}")
print(f"첫 페이지 미리보기:\n{pages[0]['text'][:500]}")

텍스트가 있는 페이지 수: 110
첫 페이지 미리보기:
2025 한국 반려동물 보고서
반려동물 건강 웰니스와 비만 관리
2025. 6
황원경 | 김남경 | 강윤정


In [12]:
chunk_size = 1000
chunk_overlap = 200

splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    separators=["\n\n", "\n", ". ", "다. ", "; ", " ", ""],
)

documents = []
for page in pages:
    page_number = page["page"]
    page_text = page["text"].strip()
    for chunk_index, chunk in enumerate(splitter.split_text(page_text)):
        documents.append({
            "text": chunk.strip(),
            "metadata": {
                "source": PDF_PATH.name,
                "source_path": str(PDF_PATH),
                "page": page_number,
                "chunk_index": chunk_index,
                "chunk_size": chunk_size,
                "chunk_overlap": chunk_overlap,
                "collection": COLLECTION_NAME,
            },
        })

assert documents, "청크가 생성되지 않았습니다. PDF 텍스트 추출 결과를 확인하세요."
print(f"생성된 청크 수: {len(documents):,}")
print(f"청크 크기: {chunk_size:,}자")
print(f"청크 오버랩: {chunk_overlap:,}자")
print(documents[0])

생성된 청크 수: 127
청크 크기: 1,000자
청크 오버랩: 200자
{'text': '2025 한국 반려동물 보고서\n반려동물 건강 웰니스와 비만 관리\n2025. 6\n황원경 | 김남경 | 강윤정', 'metadata': {'source': '2025 한국 반려동물 보고서.pdf', 'source_path': 'C:\\Users\\Playdata\\Desktop\\mle-01-p1-team2\\data\\2025 한국 반려동물 보고서.pdf', 'page': 1, 'chunk_index': 0, 'chunk_size': 1000, 'chunk_overlap': 200, 'collection': 'pet_analysis_1024'}}


## 2. 임베딩 생성 및 `pet_analysis` 컬렉션 적재

문서 내용과 페이지/청크 번호로 ID를 고정합니다. 동일한 PDF를 재실행할 때는 해당 PDF의 ID만 삭제한 뒤 다시 추가하므로 다른 컬렉션이나 문서는 영향을 받지 않습니다.

In [13]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    encode_kwargs={"normalize_embeddings": True},
)

vector_db = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embedding_model,
    persist_directory=str(CHROMA_DIR),
)

source_key = str(PDF_PATH.resolve()).encode("utf-8")
source_id = hashlib.sha256(source_key).hexdigest()[:16]
ids = [
    f"{source_id}-p{item['metadata']['page']:04d}-c{item['metadata']['chunk_index']:04d}"
    for item in documents
]

# 재실행 시 이 PDF에서 생성한 청크만 제거합니다.
existing = vector_db.get(where={"source": PDF_PATH.name}, include=[])
if existing["ids"]:
    vector_db.delete(ids=existing["ids"])

vector_db.add_texts(
    texts=[item["text"] for item in documents],
    metadatas=[item["metadata"] for item in documents],
    ids=ids,
)

print(f"적재 완료: {len(documents):,}개")
print(f"컬렉션 전체 문서 수: {vector_db._collection.count():,}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 15867.53it/s]


적재 완료: 127개
컬렉션 전체 문서 수: 127


## 3. 적재 검증

샘플 질의가 `pet_analysis` 컬렉션에서 정상적으로 검색되는지 확인합니다.

In [14]:
query = "반려동물 양육 가구의 주요 관심사"
results = vector_db.similarity_search(query, k=3)

assert len(results) == 3, f"검색 결과가 3개가 아닙니다: {len(results)}"
for index, result in enumerate(results, start=1):
    print(f"[{index}] page={result.metadata.get('page')}")
    print(result.page_content[:300].replace("\n", " ") + "\n")

[1] page=52
반려가구의양육관심사 반려가구의변치않는최대관심사는 반려동물의‘건강관리’와‘양육’이었다 반려가구의 가‘반려동물양육과관련해구체적인관심사가있다’고답한가운데건강검진·질병치료등‘건강 관리’ 관련분야가 로가장높은비율을기록했다 반려동물식사나놀이등‘양육’(45.9%) 관련분야가 위를 차지했으며 다음으로행동교정·훈련등‘교육’(23.1%), 동반가능시설정보등‘외출’(22.3%), 금융상품·양육비 용등‘자금’(21.9%) 관련분야순으로조사됐다 년조사와비교해‘건강관리’(55.0%)와‘양육’(38.8%)에대한 반려인의높은관심은변함없이지속됐고 ‘교육’ 위→ 

[2] page=8
Contents ____Ⅰ 한국 반려동물 현황 01 | 한국 반려동물 양육 현황 2 02 | 향후 양육 희망 반려동물 6 03 | 선호 품종과 입양처 8 04 | 관련 법·제도 강화 의견 12 05 | 펫티켓 성숙도 16 Key Findings 22 ____Ⅱ 반려동물의 생활 웰니스 01 | 반려동물 웰니스 인식 24 02 | 반려동물의 영양 관리 26 03 | 반려동물의 운동과 놀이 30 04 | ‘나홀로 집에’ 반려동물 케어 32 05 | 반려동물과의 여가활동 34 06 | 반려동물을 위한 건강검진 38 Key Findings 

[3] page=58
구유형별로는1인가구(78.8%), 부모자녀가구(73.7%), 부부가구(69.5%) 순으로양육지속의향이높 았고, 1인가구에서대폭증가해(+19.1%p) 만족도상승과유사한경향을보임 ◎타인에게반려동물양육을추천하겠다는반려가구는49.4%로2023년(41.9%) 대비7.5%p 증가. 가구유형별로는부모자녀가구의추천의향이가장높았으며(50.9%), 가장높은양육만족도와지속 의향을보인1인가구가추천의향역시가장큰상승폭(+10.8%p)을기록 반려가구는2023년에이어반려동물의건강검진, 질병치료와같은‘건강관리’(55.2%)와식사나놀이 등의‘양육’(45.9%) 분

